# 07 · Daily Dashboard — everything at a glance

Read-only. **Run this last**, after your daily routine:

`02_option_chain_scraper → scrape_participant_oi → 04_data_processing → fetch_underlying → 06_vol_surface → (this)`

It doesn't scrape or recompute anything — it just reads what the pipeline already produced for the latest available session and lays it out as numbers so you can form a mental picture before trading.

**Sections**
1. **Freshness** — did every layer produce data for the as-of date?
2. **Volatility & VRP snapshot** — ATM IV, realized, rich/cheap, skew, term structure, expected move, key OI levels.
3. **Term structure** — per-expiry vols + positioning.
4. **Variance risk premium** — scored history (was vol actually cheap/expensive?).
5. **Positioning** — OI walls + FII/DII/Pro/Client participant leans.

No signals or recommendations are generated here — the reads are yours to make.

In [1]:
import os, glob, math, sys
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import yaml

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 240)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

CWD = os.getcwd()
PROJECT_ROOT = os.path.dirname(CWD) if os.path.basename(CWD).lower() == 'notebooks' else CWD
if os.path.join(PROJECT_ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

from chain_parser import load_chain_csv, max_pain
import participant_oi as poi
from realized_vol import prepare

DATA = os.path.join(PROJECT_ROOT, 'data')
PROC = os.path.join(DATA, 'processed')
DM_DIR = os.path.join(PROC, 'daily_metrics')
VS = os.path.join(PROC, 'vol_surface')
SIG_DIR = os.path.join(VS, 'signals')
VRP_DIR = os.path.join(VS, 'vrp')
UND_DIR = os.path.join(DATA, 'raw', 'underlying')
POI_DIR = os.path.join(DATA, 'raw', 'nse_reports', 'participant_oi')

with open(os.path.join(PROJECT_ROOT, 'config', 'config.yaml')) as f:
    CONFIG = yaml.safe_load(f)
SYMBOLS = CONFIG['collection']['active_symbols']

def _dm_dates(sym):
    out = []
    for p in glob.glob(os.path.join(DM_DIR, f'{sym.lower()}_*.csv')):
        out.append(os.path.basename(p)[:-4].split('_', 1)[1])
    return sorted(out)

# as-of = latest session present for EVERY active symbol
_common = None
for s in SYMBOLS:
    ds = set(_dm_dates(s))
    _common = ds if _common is None else (_common & ds)
ASOF = max(_common) if _common else max(_dm_dates(SYMBOLS[0]))

def load_dm(sym, date):
    p = os.path.join(DM_DIR, f'{sym.lower()}_{date}.csv')
    return pd.read_csv(p) if os.path.isfile(p) else None

def load_sig_expiry(sym, date):
    p = os.path.join(SIG_DIR, f'{sym.lower()}_{date}.csv')
    return pd.read_csv(p) if os.path.isfile(p) else None

SIG_DAILY = pd.read_csv(os.path.join(SIG_DIR, '_daily.csv'))
VRP_DAILY = pd.read_csv(os.path.join(VRP_DIR, '_daily.csv'))
VRP_PE = pd.read_csv(os.path.join(VRP_DIR, 'per_expiry.csv'))

_UND = {}
def underlying(sym):
    if sym not in _UND:
        p = os.path.join(UND_DIR, f'{sym.lower()}.csv')
        _UND[sym] = prepare(pd.read_csv(p)) if os.path.isfile(p) else None
    return _UND[sym]

def raw_chain(sym, date, expiry):
    p = os.path.join(DATA, 'raw', sym.lower(), 'option_chain', f'{date}_{expiry}.csv')
    return load_chain_csv(p) if os.path.isfile(p) else None

def oi_walls(chain):
    piv = chain.pivot_table(index='STRIKE', columns='side', values='OI', aggfunc='sum').fillna(0.0)
    cw = float(piv['CE'].idxmax()) if 'CE' in piv.columns else float('nan')
    pw = float(piv['PE'].idxmax()) if 'PE' in piv.columns else float('nan')
    return cw, pw

def sig_daily_row(sym, date):
    r = SIG_DAILY[(SIG_DAILY.symbol == sym) & (SIG_DAILY.date == date)]
    return r.iloc[0] if len(r) else None

def vrp_daily_row(sym, date):
    r = VRP_DAILY[(VRP_DAILY.symbol == sym) & (VRP_DAILY.date == date)]
    return r.iloc[0] if len(r) else None

def day_change_pct(sym, date):
    # Only trust the underlying close/change when its latest bar IS the as-of
    # session; otherwise it's stale and the caller falls back to chain spot.
    u = underlying(sym)
    if u is None or len(u) < 2 or str(u.date.iloc[-1]) != date:
        return np.nan, np.nan
    c, p = float(u.close.iloc[-1]), float(u.close.iloc[-2])
    return c, (c / p - 1.0) * 100.0

print(f'As-of session: {ASOF}   |   symbols: {", ".join(SYMBOLS)}')

As-of session: 2026-10-07   |   symbols: NIFTY, BANKNIFTY


## 1 · Freshness
Confirm every layer produced data for the as-of session. `MISSING` means re-run that step before trusting the rest.

In [2]:
rows = []
for s in SYMBOLS:
    dm = load_dm(s, ASOF)
    sig = sig_daily_row(s, ASOF)
    vr = vrp_daily_row(s, ASOF)
    u = underlying(s)
    u_has = (u is not None) and (ASOF in set(u.date))
    rows.append({
        'symbol': s,
        'daily_metrics (04)': 'OK' if dm is not None else 'MISSING',
        'vol_signals (06)': 'OK' if sig is not None else 'MISSING',
        'vrp (06)': 'OK' if vr is not None else 'MISSING',
        'underlying@asof': 'OK' if u_has else 'stale/missing',
    })
pdate, pdf = poi.latest_report(POI_DIR, datetime.strptime(ASOF, '%Y-%m-%d'))
print('Participant OI: ' + (f'OK (report {pdate.strftime("%Y-%m-%d")})' if pdf is not None else 'MISSING'))
if pdf is not None and pdate.strftime('%Y-%m-%d') != ASOF:
    print(f'[WARN] Participant OI report is from {pdate:%Y-%m-%d}, not {ASOF}: rerun scripts/scrape_participant_oi.py')

import trading_calendar
_session = trading_calendar.resolve_session_date()
print(f'As-of: {ASOF} | {trading_calendar.describe_session()}')
if ASOF != _session:
    print(f'[WARN] Dashboard shows {ASOF}, but the latest finished NSE session is {_session}: run the pipeline for it.')
display(pd.DataFrame(rows).set_index('symbol'))

Participant OI: OK (report 2026-10-07)
As-of: 2026-10-07 | Session: 2026-10-07 (Wed) | now 2026-10-08 00:59 IST | 2026-10-08 is not a finished trading session


,daily_metrics (04),vol_signals (06),vrp (06),underlying@asof
symbol,,,,
NIFTY,OK,OK,OK,OK
BANKNIFTY,OK,OK,OK,OK


## 2 · Volatility & VRP snapshot
Front-expiry headline. IVs in vol points. `iv-rv20` > 0 → options rich vs last 20d realized; < 0 → cheap. `EM_1sig` = ±1σ expected move (points) to expiry from ATM IV; `straddle~` ≈ fair ATM straddle.

In [3]:
A, B = [], []
for s in SYMBOLS:
    dm = load_dm(s, ASOF)
    sig = sig_daily_row(s, ASOF)
    vr = vrp_daily_row(s, ASOF)
    if dm is None:
        continue
    fr = dm[dm.is_front_expiry == True]
    fr = fr.iloc[0] if len(fr) else dm.sort_values('dte').iloc[0]
    spot, chg = day_change_pct(s, ASOF)
    if not np.isfinite(spot):
        spot = float(fr.underlying_spot)
    atm_iv = float(sig.front_atm_iv) if sig is not None else np.nan
    dte = int(fr.dte)
    T = max(dte, 0) / 365.0
    em1 = spot * (atm_iv / 100.0) * math.sqrt(T) if (np.isfinite(atm_iv) and T > 0) else np.nan
    straddle = em1 * 0.7979 if np.isfinite(em1) else np.nan
    ch = raw_chain(s, ASOF, fr.expiry)
    mp = max_pain(ch)['max_pain_strike'] if ch is not None else np.nan
    cw, pw = oi_walls(ch) if ch is not None else (np.nan, np.nan)
    A.append({
        'symbol': s, 'spot': spot, 'day_%': chg, 'front': fr.expiry, 'dte': dte,
        'atm_iv': atm_iv,
        'rv20': float(vr.trailing_rv20_cc) if vr is not None else np.nan,
        'iv-rv20': float(vr.iv_minus_trail20) if vr is not None else np.nan,
        'back_iv': float(sig.back_atm_iv) if sig is not None else np.nan,
        'term_raw': float(sig.term_slope_raw) if sig is not None else np.nan,
        'rr25': float(sig.front_rr25) if sig is not None else np.nan,
        'bf25': float(sig.front_bf25) if sig is not None else np.nan,
        'no_arb': bool(sig.all_no_arb_ok) if sig is not None else None,
    })
    B.append({
        'symbol': s, 'spot': spot,
        'pcr_oi': float(fr.pcr_oi), 'pcr_vol': float(fr.pcr_volume),
        'composite_v0': float(fr.composite_v0_chg_oi_atm_band),
        'max_pain': mp, 'mp_dist_%': (mp / spot - 1) * 100 if np.isfinite(mp) else np.nan,
        'call_wall': cw, 'put_wall': pw,
        'centroid_ce': float(fr.oi_centroid_ce), 'centroid_pe': float(fr.oi_centroid_pe),
        'EM_1sig': em1, 'EM_%': (em1 / spot * 100) if np.isfinite(em1) else np.nan,
        'straddle~': straddle,
    })
print('A) Volatility & VRP  (vol pts; iv-rv20>0 = rich vs 20d realized; term_raw = back - front ATM IV)')
display(pd.DataFrame(A).set_index('symbol'))
print('B) Levels & positioning  (front expiry; walls = max-OI strikes; EM_1sig = +/-1 sigma pts to expiry)')
display(pd.DataFrame(B).set_index('symbol'))

A) Volatility & VRP  (vol pts; iv-rv20>0 = rich vs 20d realized; term_raw = back - front ATM IV)


,spot,day_%,front,dte,atm_iv,rv20,iv-rv20,back_iv,term_raw,rr25,bf25,no_arb
symbol,,,,,,,,,,,,
NIFTY,"22,603.05",-0.76,13-Oct-2026,6,11.85,12.16,-0.31,13.19,1.34,-1.18,0.30,True
BANKNIFTY,"55,055.55",-0.13,27-Oct-2026,20,14.65,13.62,1.03,13.79,-0.86,-2.08,0.35,False


B) Levels & positioning  (front expiry; walls = max-OI strikes; EM_1sig = +/-1 sigma pts to expiry)


,spot,pcr_oi,pcr_vol,composite_v0,max_pain,mp_dist_%,call_wall,put_wall,centroid_ce,centroid_pe,EM_1sig,EM_%,straddle~
symbol,,,,,,,,,,,,,
NIFTY,"22,603.05",0.81,1.05,"302,820.00","22,650.00",0.21,"23,000.00","21,500.00","23,269.75","21,816.28",343.40,1.52,274.00
BANKNIFTY,"55,055.55",0.92,0.96,"-7,072.00","55,900.00",1.53,"58,000.00","55,000.00","57,311.13","54,353.87","1,888.56",3.43,"1,506.88"


## 3 · Term structure (per expiry)
Every listed expiry for the as-of session: fitted vols (`atm_iv`, `rr25` skew, `bf25` convexity), arbitrage flag, and the positioning that sits on it (PCR, composite ΔOI, max-pain, OI walls).

In [4]:
for s in SYMBOLS:
    se = load_sig_expiry(s, ASOF)
    dm = load_dm(s, ASOF)
    print(f'=== {s}  (as-of {ASOF}) ===')
    if se is None or dm is None:
        print('  missing vol_signals or daily_metrics')
        continue
    m = se.merge(
        dm[['expiry', 'pcr_oi', 'composite_v0_chg_oi_atm_band', 'oi_centroid_ce', 'oi_centroid_pe']],
        on='expiry', how='left')
    mps, cws, pws = [], [], []
    for _, r in m.iterrows():
        ch = raw_chain(s, ASOF, r.expiry)
        if ch is not None:
            mps.append(max_pain(ch)['max_pain_strike'])
            c, p = oi_walls(ch)
            cws.append(c); pws.append(p)
        else:
            mps.append(np.nan); cws.append(np.nan); pws.append(np.nan)
    m['max_pain'], m['call_wall'], m['put_wall'] = mps, cws, pws
    cols = ['expiry', 'dte', 'atm_iv', 'rr25', 'bf25', 'no_arb_ok', 'pcr_oi',
            'composite_v0_chg_oi_atm_band', 'max_pain', 'call_wall', 'put_wall']
    out = m[cols].rename(columns={'composite_v0_chg_oi_atm_band': 'composite_v0'})
    display(out.set_index('expiry'))

=== NIFTY  (as-of 2026-10-07) ===


,dte,atm_iv,rr25,bf25,no_arb_ok,pcr_oi,composite_v0,max_pain,call_wall,put_wall
expiry,,,,,,,,,,
13-Oct-2026,6,11.85,-1.18,0.30,True,0.81,"302,820.00","22,650.00","23,000.00","21,500.00"
19-Oct-2026,12,12.40,-1.67,0.34,True,0.81,"4,313.00","22,700.00","23,500.00","20,500.00"
27-Oct-2026,20,12.89,-1.94,0.38,True,0.97,"29,286.00","23,200.00","23,500.00","23,500.00"
03-Nov-2026,27,13.11,-1.99,0.36,True,0.47,440.00,"22,700.00","24,000.00","22,000.00"
09-Nov-2026,33,13.41,-1.84,0.31,True,0.57,-23.00,"23,000.00","23,400.00","22,000.00"
23-Nov-2026,47,13.19,-1.87,0.39,True,0.86,"-6,017.00","23,000.00","24,000.00","22,500.00"


=== BANKNIFTY  (as-of 2026-10-07) ===


,dte,atm_iv,rr25,bf25,no_arb_ok,pcr_oi,composite_v0,max_pain,call_wall,put_wall
expiry,,,,,,,,,,
27-Oct-2026,20,14.65,-2.08,0.35,False,0.92,"-7,072.00","55,900.00","58,000.00","55,000.00"
23-Nov-2026,47,13.79,-2.78,0.41,True,0.62,-513.00,"57,000.00","57,000.00","57,000.00"


## 4 · Variance risk premium (scored history)
The honest scorecard: for expiries whose window has fully realized, ATM IV on the observation day vs the realized vol that actually followed. `vrp_cc` > 0 means implied was expensive (seller edge). The daily gauge at the bottom is the always-available rich/cheap read.

In [5]:
done = VRP_PE.completed.astype(str).str.lower() == 'true'
for s in SYMBOLS:
    comp = VRP_PE[done & (VRP_PE.symbol == s)].copy()
    print(f'=== {s} — VRP (completed expiry windows) ===')
    if len(comp) == 0:
        print('  no completed windows yet')
        continue
    mean_vrp = comp.vrp_cc.mean()
    win = (comp.vrp_cc > 0).mean() * 100
    print(f'  scored obs: {len(comp)}   mean VRP(cc): {mean_vrp:+.2f} vp   seller-win: {win:.0f}%')
    disp = comp.sort_values('obs_date').tail(8)[['obs_date', 'expiry', 'dte', 'atm_iv', 'rv_cc', 'vrp_cc', 'seller_win']]
    display(disp.set_index('obs_date'))
print('Daily rich/cheap gauge (front implied - trailing 20d realized):')
g = VRP_DAILY[VRP_DAILY.date == ASOF][['symbol', 'front_atm_iv', 'trailing_rv20_cc', 'iv_minus_trail20']]
display(g.set_index('symbol'))

=== NIFTY — VRP (completed expiry windows) ===
  scored obs: 139   mean VRP(cc): +1.24 vp   seller-win: 69%


,expiry,dte,atm_iv,rv_cc,vrp_cc,seller_win
obs_date,,,,,,
2026-09-23,29-Sep-2026,6,8.41,18.40,-9.99,False
2026-09-23,06-Oct-2026,13,8.50,15.50,-7.00,False
2026-09-24,29-Sep-2026,5,11.56,14.93,-3.36,False
2026-09-24,06-Oct-2026,12,11.50,13.29,-1.79,False
2026-09-28,06-Oct-2026,8,13.18,10.85,2.33,True
2026-09-29,06-Oct-2026,7,11.83,11.92,-0.09,False
2026-09-30,06-Oct-2026,6,11.76,13.21,-1.45,False
2026-10-01,06-Oct-2026,5,11.99,12.79,-0.81,False


=== BANKNIFTY — VRP (completed expiry windows) ===
  scored obs: 79   mean VRP(cc): +1.14 vp   seller-win: 62%


,expiry,dte,atm_iv,rv_cc,vrp_cc,seller_win
obs_date,,,,,,
2026-09-15,29-Sep-2026,14,14.64,15.92,-1.28,False
2026-09-16,29-Sep-2026,13,14.22,16.11,-1.89,False
2026-09-17,29-Sep-2026,12,13.17,16.93,-3.75,False
2026-09-18,29-Sep-2026,11,11.84,17.80,-5.97,False
2026-09-21,29-Sep-2026,8,11.94,19.19,-7.25,False
2026-09-22,29-Sep-2026,7,11.47,20.77,-9.30,False
2026-09-23,29-Sep-2026,6,9.99,22.74,-12.76,False
2026-09-24,29-Sep-2026,5,14.39,18.96,-4.56,False


Daily rich/cheap gauge (front implied - trailing 20d realized):


,front_atm_iv,trailing_rv20_cc,iv_minus_trail20
symbol,,,
BANKNIFTY,14.65,13.62,1.03
NIFTY,11.85,12.16,-0.31


## 5 · Positioning — participant leans
Market-wide index F&O net (long − short) contracts by participant. `net_fut_idx` > 0 = net long index futures; `d_net_fut_idx` = day-over-day change; `net_opt_idx_dir` = net long calls minus net long puts (a directional options lean). This report is aggregate across all index F&O, not per index.

In [6]:
asof_dt = datetime.strptime(ASOF, '%Y-%m-%d')
pdate, pdf = poi.latest_report(POI_DIR, asof_dt)
if pdf is None:
    print('Participant OI report not available.')
else:
    net = poi.net_positions(pdf)
    prev_date, prev_df = poi.latest_report(POI_DIR, pdate - timedelta(days=1))
    if prev_df is not None:
        pnet = poi.net_positions(prev_df)
        net = net.assign(d_net_fut_idx=net.net_fut_idx - pnet.net_fut_idx.reindex(net.index))
    else:
        net = net.assign(d_net_fut_idx=np.nan)
    show = net[['net_fut_idx', 'd_net_fut_idx', 'net_ce_idx', 'net_pe_idx', 'net_opt_idx_dir']]
    hdr = f'Participant OI (index F&O net contracts) — report {pdate.strftime("%Y-%m-%d")}'
    if prev_df is not None:
        hdr += f', d vs {prev_date.strftime("%Y-%m-%d")}'
    print(hdr)
    display(show)

Participant OI (index F&O net contracts) — report 2026-10-07, d vs 2026-10-06


,net_fut_idx,d_net_fut_idx,net_ce_idx,net_pe_idx,net_opt_idx_dir
client_type,,,,,
FII,-298285,-4560,-400079,574665,-974744
DII,33919,18,61443,54691,6752
Pro,18966,-3758,96011,85524,10487
Client,245400,8300,242626,-714880,957506
TOTAL,0,0,0,0,0


## 6 · Bates view (descriptive — context, not a signal)
Written by `python scripts/live_daily.py`. For each fitted expiry: the move the market prices to expiry (Bates, risk-neutral) vs the move our real-world model expects (GARCH, calibrated on past data only); the gap is the volatility premium in vol points (+ = options rich for sellers). Rich/cheap ranks each option only against options of the same side and delta bucket. In Stage 5 this ranking was real but worth about Rs 4 per Rs 100 of premium — below trading costs — so it is context, not a trade list.

In [7]:
LB = os.path.join(PROC, 'live_bates')
p = os.path.join(LB, '_latest.txt')
if not os.path.isfile(p):
    print('No Bates view yet - run `python scripts/live_daily.py` after 06.')
else:
    ev = pd.read_csv(os.path.join(LB, 'expiry_view.csv'), parse_dates=['date'])
    last = ev.groupby('symbol').date.max()
    for s in SYMBOLS:
        d = last.get(s)
        if d is None or str(d.date()) != ASOF:
            print(f'WARNING: Bates view for {s} is for {d.date() if d is not None else "nothing"}, not {ASOF} - re-run scripts/live_daily.py')
    print(open(p, encoding='utf-8').read())


NIFTY - 2026-10-07: Bates fit bates, 81 options / 6 expiries, fit error 0.22 vol pts (RMS)
  P model months refitted on current data: ['2026-11']
  expiry        dte  ATM IV   priced move   P-model move   gap (vol pts)   jumps/events share
  2026-10-13     6   12.08   +/- 1.88%      +/- 1.69%         +1.45          29%
  2026-10-19    12   12.42   +/- 2.70%      +/- 2.40%         +1.63          28%
  2026-10-27    20   13.07   +/- 3.54%      +/- 3.08%         +1.98          28%
  2026-11-03    27   13.04   +/- 4.16%      +/- 3.63%         +1.93          27%
  2026-11-09    33   13.29   +/- 4.63%      +/- 4.03%         +2.02          27%
  2026-11-23    47   13.09   +/- 5.60%      +/- 4.81%         +2.22          26%
  rich/cheap within side x delta bucket, expiry 2026-11-03 (S_P = ln(Bates real-world value / mid))
    PE richest per bucket: 21000 (d 0.06, mid 28.1, S_P -0.53); 21750 (d 0.16, mid 76.8, S_P -0.14); 22250 (d 0.31, mid 173.9, S_P -0.08)
    PE cheapest per bucket: 21550 (

## 7 · Paper trading — Stage 5 engine, forward test
The exact Stage 5 rules, frozen on 2026-10-07: at each monthly expiry's settlement close, sell the next monthly ATM straddle, delta-hedge daily, hold to expiry. P&L is in % of book capital (both symbols together). `replay` cycles were entered before the freeze (used to check the code against the backtest); only `forward` cycles are a true out-of-sample record. `lots/crore` shows the position in lots per Rs 1 crore of capital. **Scrape on every monthly settlement day** — a missed entry day means a skipped cycle.

In [8]:
PP = os.path.join(PROC, 'paper')
p = os.path.join(PP, '_report.txt')
if not os.path.isfile(p):
    print('No paper ledger yet - run `python scripts/live_daily.py` after 06.')
else:
    dly = pd.read_csv(os.path.join(PP, 'daily.csv'), parse_dates=['date'])
    if str(dly.date.max().date()) != ASOF:
        print(f'WARNING: paper ledger runs through {dly.date.max().date()}, not {ASOF} - re-run scripts/live_daily.py')
    txt = open(p, encoding='utf-8').read()
    print(txt.split('REPLAY vs STAGE 5')[0])
    j = os.path.join(PP, 'journal.csv')
    if os.path.isfile(j) and 'JOURNAL WARNING' in txt:
        print('JOURNAL WARNING present - see data/processed/paper/_report.txt')

PAPER ENGINE (Stage 5 monthly ATM straddle, daily hedge) - run 2026-10-07 15:28, data through 2026-10-07, code hash 471ca5d201c4
P&L in % of book capital (capital 1 = both symbols); 'lots/crore' = lots per Rs 1 crore of book capital
                       phase   status      entry final_expiry   strike  iv_entry  premium_book  lots_per_crore  gross  cost   net  net_quoted  net_stress  stale_days
pid                                                                                                                                                                  
NIFTY_2026-10-27      replay     open 2026-09-29   2026-10-27  22900.0      12.2          2.74             6.8   0.06  0.06  0.00        0.01       -0.03           0
BANKNIFTY_2026-08-25  replay  settled 2026-07-28   2026-08-25  57100.0      14.1          3.28             6.1   1.91  0.13  1.79        1.81        1.70           0
BANKNIFTY_2026-09-29  replay  settled 2026-08-25   2026-09-29  57800.0      10.9          2.65         

---
**Reading notes** — this is a snapshot, not a signal.
- `atm_iv` vs `rv20` / `iv-rv20`: how the market is pricing forward risk vs what just happened.
- `rr25` (25Δ risk-reversal): negative = puts bid over calls (downside fear). `bf25`: wings/convexity.
- `term_raw`: back minus front ATM IV — positive = upward-sloping term structure (calm front), negative = event/stress in the front.
- `max_pain` / `call_wall` / `put_wall`: OI-implied gravity and where the biggest open positions sit (soft support/resistance).
- VRP is still thin (few completed cycles) — treat it as direction, not a statistic, until weeks accumulate.